In [1]:
!pip install autogluon --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracki

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, f1_score
from sklearn.model_selection import train_test_split
import math

# Framework Library
from autogluon.tabular import TabularDataset, TabularPredictor

In [11]:
# Reading dataset
data = pd.read_csv("train.csv")
data["CabinNoCabin"] = np.where(data["Cabin"].isnull(), "No Cabin", "Cabin")
data["survived"] = np.where(data["Survived"] == 1, "yes", "no")
# data["Age"] = data["Age"].fillna(data["Age"].mean())
data.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis = 1, inplace = True)
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,CabinNoCabin
0,0,3,male,22.0,1,0,7.2500,S,No Cabin
1,1,1,female,38.0,1,0,71.2833,C,Cabin
2,1,3,female,26.0,0,0,7.9250,S,No Cabin
3,1,1,female,35.0,1,0,53.1000,S,Cabin
4,0,3,male,35.0,0,0,8.0500,S,No Cabin


In [12]:
# Split data
train_data, test_data = train_test_split(data, test_size = 0.2)

In [13]:
# Create datasets in the right format for autogluon
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)

In [14]:
# Set up framework
automl = TabularPredictor(
    label = "survived",
    # eval_metric = "accuracy, log_loss, roc_auc",
    path = "/content/drive/MyDrive/Colab Notebooks/Kaggle/agmodel_Titanic01"
)

In [15]:
# Fit model
automl.fit(train_data, time_limit=300, num_cpus=8)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sat Sep  6 09:54:41 UTC 2025
CPU Count:          2
Memory Avail:       11.04 GB / 12.67 GB (87.1%)
Disk Space Avail:   59.23 GB / 100.00 GB (59.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU memory.
	presets='best'    : Maximize accuracy. Recommended for most users. Use in 

In [16]:
# show leaderboard
automl.leaderboard(test_data, silent = True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,0.810056,0.867133,accuracy,0.014262,0.002573,2.710974,0.014262,0.002573,2.710974,1,True,5
1,LightGBM,0.810056,0.874126,accuracy,0.019521,0.011953,0.489202,0.019521,0.011953,0.489202,1,True,2
2,XGBoost,0.798883,0.888112,accuracy,0.032484,0.011087,0.541975,0.032484,0.011087,0.541975,1,True,9
3,NeuralNetFastAI,0.798883,0.867133,accuracy,0.034178,0.023531,3.412807,0.034178,0.023531,3.412807,1,True,8
4,WeightedEnsemble_L2,0.798883,0.888112,accuracy,0.038195,0.012580,0.702557,0.005711,0.001493,0.160582,2,True,12
5,NeuralNetTorch,0.793296,0.874126,accuracy,0.023683,0.013415,10.065245,0.023683,0.013415,10.065245,1,True,10
6,LightGBMXT,0.782123,0.839161,accuracy,0.021790,0.008397,9.630356,0.021790,0.008397,9.630356,1,True,1
7,LightGBMLarge,0.759777,0.853147,accuracy,0.017696,0.006234,2.880129,0.017696,0.006234,2.880129,1,True,11
8,ExtraTreesGini,0.748603,0.825175,accuracy,0.161643,0.114435,0.971434,0.161643,0.114435,0.971434,1,True,6
9,RandomForestEntr,0.737430,0.853147,accuracy,0.145086,0.101556,0.988130,0.145086,0.101556,0.988130,1,True,4


In [17]:
automl.evaluate(train_data)

{'accuracy': 0.9241573033707865,
 'balanced_accuracy': np.float64(0.9086174242424243),
 'mcc': np.float64(0.8367778942000177),
 'roc_auc': np.float64(0.9653679653679652),
 'f1': 0.8924302788844621,
 'precision': 0.9411764705882353,
 'recall': 0.8484848484848485}

In [18]:
automl.evaluate(test_data)

{'accuracy': 0.7988826815642458,
 'balanced_accuracy': np.float64(0.7882076669205382),
 'mcc': np.float64(0.588881034851506),
 'roc_auc': np.float64(0.8510408733181011),
 'f1': 0.7534246575342466,
 'precision': 0.8088235294117647,
 'recall': 0.7051282051282052}

In [23]:
print(automl.feature_importance(data))

Computing feature importance via permutation shuffling for 8 features using 891 rows with 5 shuffle sets...
	3.39s	= Expected runtime (0.68s per shuffle set)
	0.85s	= Actual runtime (Completed 5 of 5 shuffle sets)


              importance    stddev   p_value  n  p99_high   p99_low
Sex             0.204040  0.020130  0.000011  5  0.245489  0.162592
Pclass          0.102581  0.008346  0.000005  5  0.119766  0.085397
Age             0.101908  0.007751  0.000004  5  0.117868  0.085948
Fare            0.078563  0.009323  0.000023  5  0.097759  0.059368
SibSp           0.020875  0.003773  0.000123  5  0.028644  0.013107
CabinNoCabin    0.010101  0.003722  0.001863  5  0.017765  0.002437
Parch           0.005387  0.002905  0.007150  5  0.011369 -0.000594
Embarked        0.005387  0.001463  0.000594  5  0.008400  0.002374


# PREDICTION

In [24]:
# Reading new data
new_data = pd.read_csv("test.csv")
new_data["CabinNoCabin"] = np.where(new_data["Cabin"].isnull(), "No Cabin", "Cabin")
new_data.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis = 1, inplace = True)

new_data = TabularDataset(new_data)
new_data

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,CabinNoCabin
0,3,male,34.5,0,0,7.8292,Q,No Cabin
1,3,female,47.0,1,0,7.0000,S,No Cabin
2,2,male,62.0,0,0,9.6875,Q,No Cabin
3,3,male,27.0,0,0,8.6625,S,No Cabin
4,3,female,22.0,1,1,12.2875,S,No Cabin
...,...,...,...,...,...,...,...,...
413,3,male,NaN,0,0,8.0500,S,No Cabin
414,1,female,39.0,0,0,108.9000,C,Cabin
415,3,male,38.5,0,0,7.2500,S,No Cabin
416,3,male,NaN,0,0,8.0500,S,No Cabin


In [25]:
# Load saved model
model_path = "/content/drive/MyDrive/Colab Notebooks/Kaggle/agmodel_Titanic01"
predictor = TabularPredictor.load(model_path)
predictor

In [26]:
# Make predictions on the new dataset
predictions = predictor.predict_proba(new_data)
predictions

,0,1
0,0.840724,0.159276
1,0.514727,0.485273
2,0.717954,0.282046
3,0.608397,0.391603
4,0.661147,0.338853
...,...,...
413,0.974753,0.025247
414,0.016505,0.983495
415,0.984167,0.015833
416,0.974753,0.025247


In [27]:
predictions.to_csv("survived.csv")

# FINDING THE CUT-OFF POINT

In [33]:
y_true = test_data["Survived"]  # ground truth
y_proba = predictor.predict_proba(test_data)[1]  # probas for class 1

thresholds = np.linspace(0,1,101)
f1_scores = [f1_score(y_true, y_proba >= t) for t in thresholds]

best_t = thresholds[np.argmax(f1_scores)]
print("Best threshold:", best_t)

Best threshold: 0.4


In [36]:
df = pd.read_csv("test.csv")
df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S
